In [1]:
import kagglehub
from pycocotools.coco import COCO
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

path = kagglehub.dataset_download(
    "awsaf49/coco-2017-dataset",
    "coco2017/annotations/instances_val2017.json"
)

# Initialize COCO API
coco = COCO(path)

loading annotations into memory...
Done (t=0.60s)
creating index...
index created!


In [2]:
# Get category IDs to fetch subsets
category_ids = coco.getCatIds(catNms=['person', 'car', 'bicycle', 'truck', 'bus'])

# Get image Ids for these categories
# image_ids = coco.getImgIds(catIds=category_ids)
# Get all image IDs that contain any of our target classes
image_ids = set()
for category_id in category_ids:
    class_ids = coco.getImgIds(catIds=[category_id])
    image_ids.update(class_ids)
image_ids = list(image_ids)
print(len(image_ids))

# Get annotation Ids for these categories
ann_ids = coco.getAnnIds(imgIds=image_ids, catIds=category_ids)
print(len(ann_ids))

# Get categories
categories = coco.loadCats(category_ids)

2948
13952


In [3]:
# Load data into dataframes
images_df = pd.DataFrame(coco.loadImgs(image_ids)).rename(columns={'id': 'image_id'})
ann_df = pd.DataFrame(coco.loadAnns(ann_ids))
category_df = pd.DataFrame(categories).rename(columns={'id': 'category_id'})

In [4]:
# Link annotations to category names
merged = pd.merge(ann_df, category_df[['category_id', 'name', 'supercategory']], on='category_id')
# Link annotations to image metadata
final = pd.merge(merged, images_df, on='image_id')

final.head()

,segmentation,area,iscrowd,image_id,bbox,category_id,id,name,supercategory,license,file_name,coco_url,height,width,date_captured,flickr_url
0,"[[253.85, 187.23, 250.82, 193.01, 255.22, 199....",2188.08650,0,532481,"[250.82, 168.26, 70.11, 64.88]",1,508910,person,person,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
1,"[[446.65, 301.37, 437.02, 302.04, 436.02, 301....",82.66090,0,532481,"[435.35, 294.23, 13.46, 7.81]",3,1342996,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
2,"[[447.44, 295.96, 448.07, 294.22, 455.81, 293....",72.87610,0,532481,"[447.44, 293.91, 12.16, 7.65]",3,1347500,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
3,"[[472.26, 299.7, 460.59, 300.16, 461.05, 292.1...",95.94150,0,532481,"[460.59, 291.71, 12.75, 8.45]",3,1349978,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
4,"[[407.07, 295.85, 407.58, 292.31, 407.83, 290....",102.19785,0,532481,"[407.07, 287.25, 12.65, 9.86]",3,1354793,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...


In [5]:
final.shape

(13952, 16)

In [6]:
# Remove null rows
cleaned_df = final.dropna(how='any')
cleaned_df.shape

(13952, 16)

In [7]:
# Map COCO-original indices to 0-index for cleaner processing
map_ids = {old_id: index for index, old_id in enumerate(category_ids)}

# Create new column in dataframe to include 0-based indexing
cleaned_df['image_index'] = cleaned_df['category_id'].map(map_ids)

In [8]:
cleaned_df.head()

,segmentation,area,iscrowd,image_id,bbox,category_id,id,name,supercategory,license,file_name,coco_url,height,width,date_captured,flickr_url,image_index
0,"[[253.85, 187.23, 250.82, 193.01, 255.22, 199....",2188.08650,0,532481,"[250.82, 168.26, 70.11, 64.88]",1,508910,person,person,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...,0
1,"[[446.65, 301.37, 437.02, 302.04, 436.02, 301....",82.66090,0,532481,"[435.35, 294.23, 13.46, 7.81]",3,1342996,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...,2
2,"[[447.44, 295.96, 448.07, 294.22, 455.81, 293....",72.87610,0,532481,"[447.44, 293.91, 12.16, 7.65]",3,1347500,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...,2
3,"[[472.26, 299.7, 460.59, 300.16, 461.05, 292.1...",95.94150,0,532481,"[460.59, 291.71, 12.75, 8.45]",3,1349978,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...,2
4,"[[407.07, 295.85, 407.58, 292.31, 407.83, 290....",102.19785,0,532481,"[407.07, 287.25, 12.65, 9.86]",3,1354793,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...,2


In [9]:
# Rearrange to make the index column to the 0th index

cols = cleaned_df.columns.tolist()
# Remove index column from current index and insert at 0th index
cols.insert(0, cols.pop(cols.index('image_index')))
# Reassign the dataframe with the new column order
cleaned_df = cleaned_df[cols]

In [10]:
cleaned_df.head()

,image_index,segmentation,area,iscrowd,image_id,bbox,category_id,id,name,supercategory,license,file_name,coco_url,height,width,date_captured,flickr_url
0,0,"[[253.85, 187.23, 250.82, 193.01, 255.22, 199....",2188.08650,0,532481,"[250.82, 168.26, 70.11, 64.88]",1,508910,person,person,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
1,2,"[[446.65, 301.37, 437.02, 302.04, 436.02, 301....",82.66090,0,532481,"[435.35, 294.23, 13.46, 7.81]",3,1342996,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
2,2,"[[447.44, 295.96, 448.07, 294.22, 455.81, 293....",72.87610,0,532481,"[447.44, 293.91, 12.16, 7.65]",3,1347500,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
3,2,"[[472.26, 299.7, 460.59, 300.16, 461.05, 292.1...",95.94150,0,532481,"[460.59, 291.71, 12.75, 8.45]",3,1349978,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
4,2,"[[407.07, 295.85, 407.58, 292.31, 407.83, 290....",102.19785,0,532481,"[407.07, 287.25, 12.65, 9.86]",3,1354793,car,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...


In [11]:
# Delete columns
delete_columns = ['iscrowd']
cleaned_df.drop(columns=delete_columns, inplace=True)

In [12]:
# One-hot Encoding
categorical_features = ['name']
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)], remainder='passthrough', verbose_feature_names_out=False)

# Fit and transform the data
encoded_df = ct.fit_transform(cleaned_df)

# Convert back to dataframe with column names
feature_names = ct.get_feature_names_out()
cleaned_df = pd.DataFrame(encoded_df, columns=feature_names)


cleaned_df.head()


,name_bicycle,name_bus,name_car,name_person,name_truck,image_index,segmentation,area,image_id,bbox,category_id,id,supercategory,license,file_name,coco_url,height,width,date_captured,flickr_url
0,0.0,0.0,0.0,1.0,0.0,0,"[[253.85, 187.23, 250.82, 193.01, 255.22, 199....",2188.0865,532481,"[250.82, 168.26, 70.11, 64.88]",1,508910,person,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
1,0.0,0.0,1.0,0.0,0.0,2,"[[446.65, 301.37, 437.02, 302.04, 436.02, 301....",82.6609,532481,"[435.35, 294.23, 13.46, 7.81]",3,1342996,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
2,0.0,0.0,1.0,0.0,0.0,2,"[[447.44, 295.96, 448.07, 294.22, 455.81, 293....",72.8761,532481,"[447.44, 293.91, 12.16, 7.65]",3,1347500,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
3,0.0,0.0,1.0,0.0,0.0,2,"[[472.26, 299.7, 460.59, 300.16, 461.05, 292.1...",95.9415,532481,"[460.59, 291.71, 12.75, 8.45]",3,1349978,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...
4,0.0,0.0,1.0,0.0,0.0,2,"[[407.07, 295.85, 407.58, 292.31, 407.83, 290....",102.19785,532481,"[407.07, 287.25, 12.65, 9.86]",3,1354793,vehicle,3,000000532481.jpg,http://images.cocodataset.org/val2017/00000053...,426,640,2013-11-20 16:28:24,http://farm7.staticflickr.com/6048/5915494136_...


In [13]:
cleaned_df.shape

(13952, 20)

In [14]:
# Convert dataframe to CSV and save to Kaggle's working directory
cleaned_df.to_csv('/kaggle/working/filtered_test_coco_metadata.csv', index=False)